# REUTERS NEWS AGENT — VERSIÓN HÍBRIDA DDG + EVALUATOR-OPTIMIZER

**Versión:** `v0.11-ddg-function-tool`  
**Última modificación:** `2026-09-11 20:30 CEST`  
**Rama:** `fix/chapter2-news-agent`  
**Modelo:** `gpt-4.1-mini`

La prueba v0.10 confirmó que el hosted `web_search` recupera fuentes generales, pero no Reuters, mientras que DDG sí descubre Reuters. Esta versión sustituye **solo la capa de retrieval de Reuters** por un `function_tool` local basado en `ddgs` y mantiene el patrón **Evaluator-Optimizer** con `gpt-4.1-mini`.


## 1. Instalar dependencias


In [ ]:
!pip install -U openai openai-agents ddgs -q


## 2. Imports, API key y ventana temporal


In [ ]:
import json, os, re, sys, importlib.metadata as im
from dataclasses import dataclass
from datetime import datetime, timedelta, date
from urllib.parse import urlsplit, urlunsplit
from ddgs import DDGS
from google.colab import userdata
from agents import Agent, Runner
from agents.decorators import tool

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
TODAY = datetime.now().date()
START_DATE = TODAY - timedelta(days=2)

print('Python:', sys.version)
print('openai:', im.version('openai'))
print('openai-agents:', im.version('openai-agents'))
print('ddgs:', im.version('ddgs'))
print('API key presente:', bool(os.environ.get('OPENAI_API_KEY')))
print('Ventana:', START_DATE.isoformat(), '->', TODAY.isoformat())


## 3. Helpers de validación Reuters


In [ ]:
DATE_RE = re.compile(r'(20\d{2}-\d{2}-\d{2})')
URL_RE = re.compile(r'https?://[^\s)\]>]+')

def normalize_reuters_url(url: str) -> str | None:
    try:
        p = urlsplit(url)
    except Exception:
        return None
    host = (p.hostname or '').lower().rstrip('.')
    if not (host == 'reuters.com' or host.endswith('.reuters.com')):
        return None
    if p.scheme not in {'http','https'}:
        return None
    path = p.path.rstrip('/')
    if not path:
        return None
    return urlunsplit(('https', host, path + '/', '', ''))

def date_from_reuters_url(url: str) -> date | None:
    m = DATE_RE.search(urlsplit(url).path)
    if not m:
        return None
    try:
        return date.fromisoformat(m.group(1))
    except ValueError:
        return None

def in_required_window(url: str) -> bool:
    d = date_from_reuters_url(url)
    return d is not None and START_DATE <= d <= TODAY

def extract_reuters_urls(text: str) -> list[str]:
    out, seen = [], set()
    for raw in URL_RE.findall(text):
        raw = raw.rstrip('.,;')
        u = normalize_reuters_url(raw)
        if u and u not in seen:
            seen.add(u)
            out.append(u)
    return out

def requested_count(text: str, default: int = 5) -> int:
    for pat in [r'\blatest\s+(\d+)\b', r'\b(\d+)\s+(?:Reuters\s+)?articles?\b', r'\b(\d+)\s+(?:news|items?)\b']:
        m = re.search(pat, text, re.I)
        if m:
            return int(m.group(1))
    return default


## 4. Function tool: búsqueda Reuters con DDG


In [ ]:
@tool
def search_reuters(query: str, max_results: int = 10) -> str:
    """Search Reuters articles with DDG.

    Args:
        query: Topic/company/date keywords. Do not include site:reuters.com.
        max_results: Maximum DDG results to inspect.
    """
    search_query = f'site:reuters.com {query}'.strip()
    print('\n[DDG TOOL] query:', search_query)
    try:
        raw = DDGS().text(search_query, region='us-en', safesearch='off', timelimit='w', max_results=max(1,min(max_results,20)))
    except Exception as e:
        msg = {'error': f'{type(e).__name__}: {e}', 'query': search_query, 'results': []}
        print('[DDG TOOL] ERROR:', msg['error'])
        return json.dumps(msg, ensure_ascii=False)

    results, seen = [], set()
    for item in raw or []:
        href = item.get('href') or item.get('url') or ''
        url = normalize_reuters_url(href)
        if not url or url in seen:
            continue
        article_date = date_from_reuters_url(url)
        if article_date is None or not (START_DATE <= article_date <= TODAY):
            continue
        seen.add(url)
        results.append({'title': item.get('title') or '', 'url': url, 'publication_date': article_date.isoformat(), 'snippet': item.get('body') or item.get('snippet') or ''})

    print(f'[DDG TOOL] valid Reuters results in window: {len(results)}')
    for r in results:
        print('  ', r['publication_date'], '-', r['title'])
        print('     ', r['url'])
    return json.dumps({'query': search_query, 'window_start': START_DATE.isoformat(), 'window_end': TODAY.isoformat(), 'results': results}, ensure_ascii=False)


## 5. Searcher y Evaluator


In [ ]:
SEARCHER_INSTRUCTIONS = f'''
You are a financial-news research agent.
Reuters discovery MUST be done with the local search_reuters tool.
Do not use memory as evidence. Do not invent headlines, dates, summaries, or URLs.
The allowed publication window is {START_DATE.isoformat()} through {TODAY.isoformat()}, inclusive.
Only direct reuters.com URLs returned by the tool are valid. Copy URLs exactly from tool output.
If the first tool call is insufficient, call the tool again with different topic/company keywords.
Search requested companies/topics separately when useful. Prefer newest items and remove duplicates.
Market quotes, ETF/index pages and generic stock-price pages do not count.
For every final item provide headline, publication date, short summary based only on title/snippet, Publisher Reuters, and direct Reuters URL.
Return the exact number requested if enough verified tool results exist. Otherwise return only verified items.
'''

web_news_searcher = Agent(name='web_news_searcher', model='gpt-4.1-mini', instructions=SEARCHER_INSTRUCTIONS, tools=[search_reuters])

@dataclass
class EvaluationFeedback:
    feedback: str
    score: str

EVALUATOR_INSTRUCTIONS = f'''
You are a strict evaluator of a Reuters financial-news result.
The allowed date window is {START_DATE.isoformat()} through {TODAY.isoformat()}, inclusive.
You receive the original request, generated answer, requested count, valid Reuters URL count and valid Reuters URLs.
Successful only if the answer contains at least the requested number of distinct genuine Reuters URLs, all inside the date window, and each item has headline/date/summary/Publisher Reuters/direct URL and is relevant to the original request.
Zero results or too few results is always unsuccessful. Never declare success merely because the answer honestly says no results were found.
Return concise actionable feedback.
'''

news_evaluator = Agent(name='news_evaluator', model='gpt-4.1-mini', instructions=EVALUATOR_INSTRUCTIONS, output_type=EvaluationFeedback)


## 6. Evaluator-Optimizer loop


In [ ]:
async def main() -> None:
    msg = input("User's request: " ).strip()
    target_count = requested_count(msg, default=5)
    print('\nRequested count:', target_count)
    print('Date window:', START_DATE.isoformat(), '->', TODAY.isoformat())

    max_iterations = 4
    feedback = None
    latest_answer = ''

    for iteration in range(1, max_iterations + 1):
        print('\n\033[92m' + f'************************** NEWS SEARCH {iteration} **************************' + '\033[0m')
        if feedback is None:
            search_prompt = msg
        else:
            search_prompt = f"{msg}\n\nThe previous answer failed evaluation.\nEvaluator feedback: {feedback}\n\nRun NEW Reuters searches with the search_reuters tool. Use different company/topic keywords where necessary. Do not reuse unsupported claims."

        search_result = await Runner.run(web_news_searcher, search_prompt)
        latest_answer = str(search_result.final_output)
        print('\nGENERATED ANSWER:\n')
        print(latest_answer)

        urls = extract_reuters_urls(latest_answer)
        valid_urls = [u for u in urls if in_required_window(u)]
        print('\n\033[93m************************** DETERMINISTIC VALIDATION **************************\033[0m')
        print('Reuters URLs in answer:', len(urls))
        print('Valid Reuters URLs in date window:', len(valid_urls))
        for u in valid_urls:
            print('  VALID:', u)

        evaluator_input = f"ORIGINAL REQUEST:\n{msg}\n\nGENERATED ANSWER:\n{latest_answer}\n\nREQUESTED COUNT: {target_count}\nVALID REUTERS URL COUNT: {len(valid_urls)}\nVALID REUTERS URLS:\n" + ('\n'.join(valid_urls) if valid_urls else 'NONE')
        print('\n\033[92m************************** RUNNING EVALUATION **************************\033[0m')
        eval_result = await Runner.run(news_evaluator, evaluator_input)
        result: EvaluationFeedback = eval_result.final_output

        if len(valid_urls) < target_count:
            result.score = 'unsuccessful'
            result.feedback = f'Only {len(valid_urls)} valid Reuters URLs were present; {target_count} are required. Search again with new Reuters queries.'

        print('Evaluator score:', result.score)
        print('Evaluator feedback:', result.feedback)
        if result.score == 'successful':
            print('Evaluation successful ==> stopping iteration.')
            break
        feedback = result.feedback
        if iteration == max_iterations:
            print('Reached max_iterations ==> stopping iteration.')

    print('\n\033[92m************************** FINAL NEWS SET **************************\033[0m')
    print(latest_answer)


## 7. Ejecutar


In [ ]:
await main()
